In [8]:
import cv2
from hand_tracker import HandDetector
from utils import load_overlay_img, overlay_transparent

# 1. 사용할 사진 파일의 경로 지정
img_a_path = 'cat img1.webp'  # 왼손 주먹용
img_b_path = 'cat img2.jpg'  # 오른손 주먹용
img_c_path = 'cat img3.jpg'  # 손가락 걷기용 (새로 추가)

# 2. 화면에 띄울 이미지 크기 지정
IMG_WIDTH = 150
IMG_HEIGHT = 150

# 이미지 로드 (없으면 임시 네모 상자가 나타남)
img_a = load_overlay_img(img_a_path, size=(IMG_WIDTH, IMG_HEIGHT))
img_b = load_overlay_img(img_b_path, size=(IMG_WIDTH, IMG_HEIGHT))
img_c = load_overlay_img(img_c_path, size=(IMG_WIDTH, IMG_HEIGHT))

# 3. 인식 모델 초기화
detector = HandDetector()
cap = cv2.VideoCapture(0)

print("시작하려면 카메라 창을 클릭하고 'q'를 누르세요.")

# --- 메인 실행 루프 ---
while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    
    frame = cv2.flip(frame, 1) # 좌우 반전
    results = detector.find_hands(frame)
    hands_info = detector.get_hand_info(frame, results)
    
    for hand in hands_info:
        cx, cy = hand["center"]
        # 사진이 출력될 기본 좌표 계산 (손목 중심점 위쪽)
        out_x = cx - (IMG_WIDTH // 2)
        out_y = cy - IMG_HEIGHT - 20
        
        # [중요] 우선순위 로직: 손가락 걷기 이벤트가 무조건 우선!
        if hand["is_walking"]:
            # 걷기 이벤트 발생 시 양손 구분 없이 이미지 C 출력
            frame = overlay_transparent(frame, img_c, out_x, out_y)
            
        elif hand["is_fist"]:
            # 걷기 상태가 아닐 때만 주먹 확인 (왼손/오른손 구분)
            if hand["label"] == "Left":
                frame = overlay_transparent(frame, img_a, out_x, out_y)
            elif hand["label"] == "Right":
                frame = overlay_transparent(frame, img_b, out_x, out_y)
                
    cv2.imshow("Hand Control System", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

시작하려면 카메라 창을 클릭하고 'q'를 누르세요.
